In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
criminals_path = '/content/drive/MyDrive/criminals/'

# Check if it exists
if os.path.exists(criminals_path):
    print("✓ Found criminals folder!")
    files = os.listdir(criminals_path)
    print(f"  Contains {len(files)} files")
else:
    print("✗ Folder not found. Creating it...")
    os.makedirs(criminals_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Found criminals folder!
  Contains 3316 files


In [2]:
!pip install transformers
!pip install faiss-cpu
!pip install faiss-gpu
!pip install -U bitsandbytes
!pip install qwen_vl_utils
!pip install pandas
!pip install  torchvision
!pip install accelerate
!pip install chromadb

ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


In [3]:
import torch
import sklearn
from torch import nn
from torchvision import transforms
from PIL import Image

In [12]:
import re
def preprocess_text(result):
# Your original text


# Option 1: Get the matched text and convert to lowercase
  matches = re.search(r'assistant\s*:\s*(.*)', result, re.IGNORECASE)

  if matches:
    # Group 1 contains the text after "assistant"
    final_answer = matches.group(1).strip().lower()  # .group(1) extracts the captured part
  else:
    final_answer = "none"
  return final_answer


In [5]:
def answer_to_number(results):
  for i in range(len(results)):
     if results[i] == "yes" or results[i] == "yes.":
       results[i] = 1
     elif results[i] == "no" or results[i] == "no.":
       results[i] = 0
     else :
       results[i] = -1
  return results
def computation(labels,results):
  FN,TN,FP,TP,accur = 0,0,0,0,0
  for i in range(len(labels)):
     if labels[i] == 1 and results[i] == 1:
       TP += 1
     elif labels[i] == 1 and results[i] == 0:
       FN += 1
     elif labels[i] == 0 and results[i] == 1:
       FP += 1
     elif labels[i] == 0 and results[i] == 0:
       TN += 1
     else:
       continue
  for i in range(len(labels)):
    if labels[i] == results[i]:
      accur += 1
  accuracy = accur/len(labels)
  LR_PLUS = (TP/(TP+FN))/(FP/(FP+TN))
  LR_MINUS = (FN/(TP+FN))/(TN/(FP+TN))
  NPV = TN/(TN+FN)
  answer = {
      "LR+":LR_PLUS,
      "LR-":LR_MINUS,
      "NPV":NPV,
      "accuracy":accuracy
  }
  return answer
def collection(results):
  combo = {"yes":0,"no":0,"others":0}
  for i in range(len(results)):
    if results[i] == "yes" or results[i] == "yes.":
      combo["yes"] += 1
    elif results[i] == "no" or results[i] == "no.":
      combo["no"] += 1
    else:
      combo["others"] += 1
  return combo

In [7]:
from transformers import BitsAndBytesConfig,AutoProcessor
from transformers import Idefics3ForConditionalGeneration
processor_idefics = AutoProcessor.from_pretrained("HuggingFaceM4/Idefics3-8B-Llama3")
model_idefics = Idefics3ForConditionalGeneration.from_pretrained(
    "HuggingFaceM4/Idefics3-8B-Llama3",
    torch_dtype=torch.float16,
    device_map="auto",
)
model_idefics.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

Idefics3ForConditionalGeneration(
  (model): Idefics3Model(
    (vision_model): Idefics3VisionTransformer(
      (embeddings): Idefics3VisionEmbeddings(
        (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
        (position_embedding): Embedding(676, 1152)
      )
      (encoder): Idefics3Encoder(
        (layers): ModuleList(
          (0-26): 27 x Idefics3EncoderLayer(
            (self_attn): Idefics3VisionAttention(
              (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
            )
            (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (mlp): Idefics3VisionMLP(
              (activation_fn): GELUTanh()
              (fc1): Linear(in_feature

In [8]:
import pandas as pd
df1 = pd.read_csv('train_offense_facts.csv', on_bad_lines='skip')
df2 = pd.read_csv('test_preprocessed_with_images_and_caste (1).csv', on_bad_lines='skip')
df1 = df1[['id','label','only_facts']]
df2 = df2[['id','label','facts_and_arguments']]

argument_keywords = [
    'hence',
    'oppose',
    'opposes',
    'opposed',
    'opposing',
    'support',
    'supports',
    'supported',
    'supporting',
    'bailable',
    'granted',
    'rejected'
]

only_facts = []
for fact_arg in df2['facts_and_arguments']:
    sents = fact_arg.split('. ')
    new_sents = []
    for s in sents:
        flag = True
        for key in argument_keywords:
            if key in s:
                flag = False
                break
        if flag:
          new_sents.append(s)
    only_facts.append('. '.join(new_sents))
df2.loc[:, 'only_facts'] = only_facts


In [9]:
print(df2)

                                             id  label  \
0      Bail Application_2180_202002-01-20211157      0   
1       Bail Application_1017_202006-07-2020391      1   
2      Bail Application_1156_202122-02-20215574      1   
3     Bail Application_101049_202131-03-2021293      1   
4      Bail Application_4458_202006-10-20202515      1   
...                                         ...    ...   
3311  Bail Application__1545_202112-03-20211846      1   
3312           Bail Appl__4218_201920-12-201970      0   
3313    Bail Application_750_202105-03-20211151      0   
3314    Bail Application_584_202102-02-20212940      0   
3315     Bail Application_321_202017-02-2020527      1   

                                    facts_and_arguments  \
0     When the plaintiff Kibahan told the above thin...   
1     According to the prosecution, the inspector-in...   
2     The accused is in judicial custody. The learne...   
3     The investigator has compiled sufficient again...   
4     Ac

In [18]:
results2 = []
labels = [df2['label'].iloc[i] for i in range(len(df2))]
for i in range(len(df2)):
 test_img_path = f"/content/drive/MyDrive/criminals/{i:05d}.jpg"
 image = Image.open(test_img_path)
 image = image.resize((224, 224), Image.BICUBIC)
 image = image.convert("RGB")
 text = df2['only_facts'].iloc[i]
 label = df2['label'].iloc[i]

 system_prompt = '''Instruction: You are provided with the facts of a criminal case report.
                    You need to understand how a bail decision is made from the facts in order to respond to the question.'''
 user_prompt = f'''Question: The facts of the current case are the following: {text}
                  You are also given an image of the accused person which is the photo of the accused which you need to see.
                  Should the accused person be granted bail? Please answer in yes/no only.
                  \n Answer:'''

 conversation = [
     {
         "role": "system",
         "content": system_prompt
     },
    {
        "role": "user",
        "content": [
            {"type": "image", "image" : image},
            {"type": "text", "text": user_prompt}
        ]
    }
]
 text = processor_idefics.apply_chat_template(conversation, add_generation_prompt=True)
 inputs = processor_idefics(images=image, text=text, return_tensors="pt")
 inputs = inputs.to("cuda")
 generated_ids = model_idefics.generate(**inputs, return_dict_in_generate=True,
                                         output_scores=True,
                                         do_sample=True,
                                         max_new_tokens=256,
                                         temperature=0.1)
 answer_text = processor_idefics.tokenizer.batch_decode(generated_ids.sequences, skip_special_tokens=True)
 answer_text = answer_text[0].strip()

 print(i+1)

 ans = preprocess_text(answer_text)
 print(ans)
 results2.append(ans)

Streaming output truncated to the last 5000 lines.
817
no.
818
yes.
819
no.
820
no.
821
yes.
822
no.
823
no.
824
no.
825
no.
826
no.
827
no.
828
no.
829
no.
830
no.
831
no.
832
no.
833
no.
834
no.
835
yes.
836
no.
837
yes.
838
yes.
839
no.
840
yes.
841
no.
842
yes.
843
no.
844
no.
845
no.
846
no.
847
yes.
848
no.
849
no.
850
no.
851
yes.
852
no.
853
no.
854
no.
855
no.
856
no.
857
no.
858
yes.
859
no.
860
no.
861
no.
862
no.
863
no.
864
no.
865
no.
866
no.
867
no.
868
no.
869
yes.
870
no.
871
no.
872
no.
873
no.
874
no.
875
no.
876
yes.
877
no.
878
no.
879
no.
880
no.
881
yes.
882
no.
883
no.
884
no.
885
yes.
886
no.
887
no.
888
no.
889
no.
890
no.
891
no.
892
no.
893
yes.
894
yes.
895
no.
896
yes.
897
no.
898
no.
899
no.
900
no.
901
no.
902
yes.
903
no.
904
no.
905
no.
906
no.
907
no.
908
no.
909
yes.
910
no.
911
no.
912
no.
913
yes.
914
no.
915
yes.
916
yes.
917
no.
918
yes.
919
no.
920
no.
921
no.
922
no.
923
no.
924
no.
925
no.
926
no.
927
yes.
928
yes.
929
yes.
930
no.
931
no.
932

In [19]:
print(results2)

['no.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'yes.', 'no.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'no.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'yes.', 'no.', 'yes.', 'no.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'no.', 'yes.', 'yes.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'yes.', 'yes.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'no.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'yes.', 'yes.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'no.', 'yes.', 'no.', 'yes.', 'no.', 'no.', 'no.', 'no.', 'no.', 'yes.', 'yes.', 'no.

In [20]:
for i in range(len(results2)):
  matches = re.search(r'\b(yes|no)\b', results2[i], re.IGNORECASE)

  if matches:
    results2[i] = matches.group(1).lower()
  else:
    results2[i] = "none"
print(results2)

['no', 'no', 'yes', 'no', 'no', 'no', 'yes', 'no', 'no', 'yes', 'no', 'no', 'no', 'no', 'no', 'yes', 'no', 'no', 'no', 'no', 'no', 'no', 'yes', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'yes', 'no', 'no', 'no', 'yes', 'no', 'no', 'no', 'yes', 'no', 'yes', 'no', 'no', 'yes', 'no', 'no', 'no', 'no', 'yes', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'yes', 'no', 'no', 'no', 'no', 'yes', 'yes', 'no', 'yes', 'no', 'no', 'no', 'no', 'yes', 'no', 'no', 'no', 'yes', 'yes', 'no', 'yes', 'no', 'no', 'no', 'no', 'no', 'yes', 'no', 'no', 'no', 'yes', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'yes', 'no', 'no', 'no', 'no', 'no', 'no', 'yes', 'yes', 'no', 'no', 'no', 'no', 'no', 'no', 'yes', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'yes', 'no', 'yes', 'no', 'no', 'no', 'no', 'no', 'yes', 'yes', 'no', 'no', 'yes', 'no', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'yes', 'no', 'no', 'no', 'no', 'no'

In [21]:


print("Without RAG:")
print()
print(collection(results2))
results2 = answer_to_number(results2)
print(labels)
print(results2)
print(computation(labels,results2))

Without RAG:

{'yes': 771, 'no': 2545, 'others': 0}
[np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0)

In [22]:
!pip install sentence_transformers
!pip install rank_bm25

In [23]:
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import numpy as np
client = chromadb.Client()
collection = client.create_collection(name="docs", get_or_create=True)

embedder = SentenceTransformer("all-MiniLM-L6-v2").cuda()
docs = [
    df1['only_facts'].iloc[i]  for i in range(len(df1))
]

embeddings = embedder.encode(docs).tolist()

# Split data into smaller batches to avoid exceeding ChromaDB's batch size limit
batch_size = 5000 # Using 5000, which is less than the max_batch_size of 5461
for i in range(0, len(docs), batch_size):
    batch_docs = docs[i:i + batch_size]
    batch_embeddings = embeddings[i:i + batch_size]
    batch_ids = [f"{j}" for j in range(i, min(i + batch_size, len(docs)))]

    collection.add(
        documents=batch_docs,
        embeddings=batch_embeddings,
        ids=batch_ids
    )

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
def create_message(index,combined_results,combined_labels):
  return f'''case{index+1}:{combined_results[index]}\n
             Based on the facts the final decision of the case is that the bail was {'granted' if combined_labels[index]==1 else 'rejected'}.'''

In [26]:
results1 = []
labels = [df2['label'].iloc[i] for i in range(len(df2))]
for i in range(len(df2)):
  test_img_path = f"/content/drive/MyDrive/criminals/{i:05d}.jpg"
  image = Image.open(test_img_path)
  image = image.resize((224, 224), Image.BICUBIC)
  image = image.convert("RGB")
  results_chroma = collection.query(
    query_embeddings=embedder.encode(df2['only_facts'].iloc[i]).tolist(),
    n_results=3,
    include=["documents", "distances", "metadatas"]
)

  documents = results_chroma['documents'][0]
  sim_score = results_chroma['distances'][0]
  index=results_chroma['ids'][0]
  index = [int(index[i]) for i in range(len(index))]

  combined_labels = [df1['label'].iloc[i] for i in index]

  text = df2['only_facts'].iloc[i]
  label = df2['label'].iloc[i]
  user_prompt = '''You are a bail decision assistant. Study these example cases and apply the pattern to the new case.

EXAMPLE CASES (learn from these):

'''

# Add examples in CLEAR pattern
  for j in range(len(documents)):
    user_prompt += f'''Case {j+1}:
Facts: {documents[j]}
Decision: {"GRANT BAIL (yes)" if combined_labels[j] == 1 else "DENY BAIL (no)"}

'''

  user_prompt += f'''NEW CASE TO DECIDE:
Facts: {text}

INSTRUCTIONS:
1. Compare this new case with the 3 example cases above
2. Look for SIMILAR factors between cases
3. If the new case is similar to majority of the example cases where bail was GRANTED, answer "yes"
4. If the new case is similar to majority of the example cases where bail was DENIED, answer "no"

Important: You MUST base your decision on the comparison with examples.

Decision (yes/no only): '''
  conversation = [

    {
        "role": "user",
        "content": [
            {"type": "image", "image" : image},
            {"type": "text", "text": user_prompt}
        ]
    }
]
  text = processor_idefics.apply_chat_template(conversation, add_generation_prompt=True)
  inputs = processor_idefics(images=image, text=text, return_tensors="pt")
  inputs = inputs.to("cuda")
  generated_ids = model_idefics.generate(**inputs, return_dict_in_generate=True,
                                         output_scores=True,
                                         do_sample=True,
                                         max_new_tokens=256,
                                         temperature=0.1)
  answer_text = processor_idefics.tokenizer.batch_decode(generated_ids.sequences, skip_special_tokens=True)
  answer_text = answer_text[0].strip()
  print(i+1)
  ans = preprocess_text(answer_text)
  print(ans)
  results1.append(ans)


Streaming output truncated to the last 5000 lines.
817
no.
818
yes.
819
no.
820
no.
821
yes.
822
no.
823
yes.
824
yes.
825
yes.
826
yes.
827
no.
828
yes.
829
no.
830
no.
831
no.
832
no.
833
yes.
834
no.
835
yes.
836
no.
837
yes.
838
yes.
839
yes.
840
yes.
841
yes.
842
yes.
843
yes.
844
no.
845
no.
846
no.
847
yes.
848
yes.
849
no.
850
no.
851
yes.
852
yes.
853
yes.
854
yes.
855
yes.
856
no.
857
no.
858
yes.
859
yes.
860
no.
861
yes.
862
no.
863
yes.
864
yes.
865
no.
866
no.
867
yes.
868
no.
869
yes.
870
yes.
871
no.
872
no.
873
yes.
874
no.
875
yes.
876
yes.
877
yes.
878
yes.
879
yes.
880
no.
881
no.
882
no.
883
no.
884
no.
885
yes.
886
yes.
887
yes.
888
yes.
889
yes.
890
yes.
891
no.
892
no.
893
yes.
894
yes.
895
yes.
896
yes.
897
no.
898
yes.
899
no.
900
no.
901
yes.
902
yes.
903
no.
904
yes.
905
no.
906
yes.
907
yes.
908
no.
909
yes.
910
yes.
911
no.
912
yes.
913
yes.
914
no.
915
yes.
916
no.
917
no.
918
yes.
919
no.
920
yes.
921
no.
922
no.
923
no.
924
no.
925
no.
926
yes.
927
yes.

In [27]:
print(results1)

['yes.', 'no.', 'yes.', 'no.', 'yes.', 'yes.', 'yes.', 'yes.', 'yes.', 'yes.', 'no.', 'yes.', 'no.', 'no.', 'yes.', 'yes.', 'no.', 'no.', 'no.', 'yes.', 'yes.', 'no.', 'yes.', 'no.', 'no.', 'yes.', 'no.', 'no.', 'yes.', 'yes.', 'no.', 'yes.', 'yes.', 'no.', 'yes.', 'yes.', 'yes.', 'no.', 'yes.', 'yes.', 'yes.', 'no.', 'yes.', 'yes.', 'no.', 'yes.', 'no.', 'yes.', 'yes.', 'yes.', 'yes.', 'no.', 'no.', 'no.', 'yes.', 'no.', 'yes.', 'yes.', 'yes.', 'yes.', 'no.', 'yes.', 'yes.', 'yes.', 'yes.', 'yes.', 'yes.', 'no.', 'no.', 'yes.', 'no.', 'yes.', 'no.', 'yes.', 'yes.', 'yes.', 'no.', 'yes.', 'yes.', 'yes.', 'yes.', 'no.', 'yes.', 'yes.', 'yes.', 'yes.', 'no.', 'yes.', 'no.', 'yes.', 'yes.', 'yes.', 'yes.', 'yes.', 'yes.', 'yes.', 'no.', 'yes.', 'yes.', 'no.', 'no.', 'yes.', 'no.', 'yes.', 'yes.', 'no.', 'yes.', 'no.', 'yes.', 'no.', 'yes.', 'yes.', 'yes.', 'no.', 'no.', 'yes.', 'yes.', 'no.', 'yes.', 'no.', 'yes.', 'no.', 'no.', 'yes.', 'no.', 'yes.', 'yes.', 'yes.', 'yes.', 'yes.', 'yes.

In [28]:
def answer_to_number(results):
  for i in range(len(results)):
     if results[i] == "yes" or results[i] == "yes.":
       results[i] = 1
     elif results[i] == "no" or results[i] == "no.":
       results[i] = 0
     else :
       results[i] = -1
  return results
def computation(labels,results):
  FN,TN,FP,TP,accur = 0,0,0,0,0
  for i in range(len(labels)):
     if labels[i] == 1 and results[i] == 1:
       TP += 1
     elif labels[i] == 1 and results[i] == 0:
       FN += 1
     elif labels[i] == 0 and results[i] == 1:
       FP += 1
     elif labels[i] == 0 and results[i] == 0:
       TN += 1
     else:
       continue
  for i in range(len(labels)):
    if labels[i] == results[i]:
      accur += 1
  accuracy = accur/len(labels)
  LR_PLUS = (TP/(TP+FN))/(FP/(FP+TN))
  LR_MINUS = (FN/(TP+FN))/(TN/(FP+TN))
  NPV = TN/(TN+FN)
  answer = {
      "LR+":LR_PLUS,
      "LR-":LR_MINUS,
      "NPV":NPV,
      "accuracy":accuracy
  }
  return answer
def collection(results):
  combo = {"yes":0,"no":0,"others":0}
  for i in range(len(results)):
    if results[i] == "yes" or results[i] == "yes.":
      combo["yes"] += 1
    elif results[i] == "no" or results[i] == "no.":
      combo["no"] += 1
    else:
      combo["others"] += 1
  return combo

In [29]:
def processor(results):
 for i in range(len(results)):
   matches = re.findall(r'\b(yes|no)\b', results[i], flags=re.IGNORECASE)
   results[i] = matches[-1].lower() if matches else "None"
 return results

In [30]:


results1 = processor(results1) #2 nd order preprocessing
print("With RAG:")
print(collection(results1))
results1 = answer_to_number(results1)
print(labels)
print(results1)
print(computation(labels,results1))

With RAG:
{'yes': 2026, 'no': 1290, 'others': 0}
[np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), n